# Pandas: Grouping, Aggregating, and Joining

This notebook covers:
1. `groupby()` and how it splits data into pieces that can be summarized
2. Common aggregations: `mean`, `sum`, `count`, and the flexible `agg()`
3. Running multiple aggregations at once, and writing custom ones
4. Pivot tables and crosstabs
5. Reshaping between wide and long format with `melt` and `pivot`
6. Joining DataFrames the way SQL does
7. Stacking DataFrames with `concat`






In [1]:
import pandas as pd
import numpy as np
np.random.seed(42)

# Build a sample sales dataset
n = 200
sales = pd.DataFrame({
    'order_id':   range(1, n + 1),
    'date':       pd.date_range('2024-01-01', periods=n, freq='D'),
    'region':     np.random.choice(['North', 'South', 'East', 'West'], n),
    'product':    np.random.choice(['Laptop', 'Phone', 'Tablet', 'Monitor', 'Headphones'], n),
    'salesperson':np.random.choice(['Ali', 'Sara', 'Ahmed', 'Fatima', 'Hassan'], n),
    'quantity':   np.random.randint(1, 10, n),
    'unit_price': np.random.choice([300, 500, 800, 1200, 150], n),
})
sales['revenue'] = sales['quantity'] * sales['unit_price']

print(sales.head(10))
print(f"\nShape: {sales.shape}")

   order_id       date region     product salesperson  quantity  unit_price  \
0         1 2024-01-01   East      Laptop         Ali         5         800   
1         2 2024-01-02   West     Monitor         Ali         4         150   
2         3 2024-01-03  North  Headphones       Ahmed         5         500   
3         4 2024-01-04   East     Monitor      Hassan         7         150   
4         5 2024-01-05   East  Headphones        Sara         9         500   
5         6 2024-01-06   West  Headphones        Sara         7         800   
6         7 2024-01-07  North      Tablet       Ahmed         5         800   
7         8 2024-01-08  North  Headphones        Sara         7         150   
8         9 2024-01-09   East     Monitor         Ali         5         150   
9        10 2024-01-10  South  Headphones      Hassan         3         500   

   revenue  
0     4000  
1      600  
2     2500  
3     1050  
4     4500  
5     5600  
6     4000  
7     1050  
8      750  


## `groupby()` - Split, Apply, Combine

### definition

> "A groupby operation involves some combination of splitting the object, applying a function, and combining the results. This can be used to group large amounts of data and compute operations on these groups." - pandas docs

A few terms before diving in:
- **Group**: a subset of rows that share the same value in one or more columns (for example, all orders from the same region).
- **Aggregation**: a function that takes a group of values and reduces them to a single value (sum, mean, count, max, etc.).
- **Split-apply-combine**: the three-step pattern groupby is built on.

The three steps:

1. **Split** the rows into groups based on the values in one or more columns.
2. **Apply** a function to each group separately (usually a summary like mean or sum).
3. **Combine** the per-group results back into a single Series or DataFrame.

### Basic groupby and aggregation

In [2]:
# Total revenue per region
revenue_by_region = sales.groupby('region')['revenue'].sum()
print("Total revenue per region:")
print(revenue_by_region)
print(f"\nType: {type(revenue_by_region).__name__}")

Total revenue per region:
region
East     155550
North    110850
South    122750
West     154200
Name: revenue, dtype: int32

Type: Series


In [3]:
# Mean revenue per product
print("Average revenue per product:")
print(sales.groupby('product')['revenue'].mean().round(2))

Average revenue per product:
product
Headphones    3128.33
Laptop        2403.92
Monitor       2689.13
Phone         2636.21
Tablet        2880.68
Name: revenue, dtype: float64


In [4]:
# Multiple aggregations at once on the same column
print("Revenue stats per region:")
print(sales.groupby('region')['revenue'].agg(['sum', 'mean', 'min', 'max', 'count']).round(2))

Revenue stats per region:
           sum     mean  min    max  count
region                                    
East    155550  2880.56  150  10800     54
North   110850  2409.78  450   9600     46
South   122750  2668.48  150   9600     46
West    154200  2855.56  150  10800     54


### Grouping by multiple columns

Passing a list of column names to `groupby` forms groups based on the combination of all those columns (every unique pair of `region` and `product` becomes its own group, for example).

In [5]:
# Total revenue by region AND product
result = sales.groupby(['region', 'product'])['revenue'].sum()
print(result)

region  product   
East    Headphones    31150
        Laptop        35050
        Monitor       37750
        Phone         27150
        Tablet        24450
North   Headphones    33100
        Laptop        16600
        Monitor       16400
        Phone          4550
        Tablet        40200
South   Headphones     5700
        Laptop        32850
        Monitor       36000
        Phone         21850
        Tablet        26350
West    Headphones    23900
        Laptop        38100
        Monitor       33550
        Phone         22900
        Tablet        35750
Name: revenue, dtype: int32


The result is a Series with a **MultiIndex** (two levels here: region and product).

### definition

> "MultiIndex objects are the hierarchical analog of the standard Index object which typically stores the axis labels in pandas objects." - pandas docs

A MultiIndex stores more than one label per row, stacked into levels. When the output gets long because of nested grouping, `.unstack()` moves the inner level into columns to make the table easier to read.

In [6]:
# Pivot the inner level (product) into columns
result_wide = sales.groupby(['region', 'product'])['revenue'].sum().unstack()
print(result_wide)

product  Headphones  Laptop  Monitor  Phone  Tablet
region                                             
East          31150   35050    37750  27150   24450
North         33100   16600    16400   4550   40200
South          5700   32850    36000  21850   26350
West          23900   38100    33550  22900   35750


### Aggregating multiple columns at once

`groupby` can summarize several columns in a single call. Pass a list of column names after the groupby, then apply the aggregation. For different aggregations on different columns, pass a dictionary to `agg()`.

In [7]:
# Sum quantity AND revenue per region
print(sales.groupby('region')[['quantity', 'revenue']].sum())

        quantity  revenue
region                   
East         275   155550
North        229   110850
South        226   122750
West         233   154200


In [8]:
# Different functions for different columns using agg() with a dict
print(sales.groupby('region').agg({
    'quantity': 'sum',
    'revenue':  ['mean', 'max'],
    'order_id': 'count'
}).round(2))

       quantity  revenue        order_id
            sum     mean    max    count
region                                  
East        275  2880.56  10800       54
North       229  2409.78   9600       46
South       226  2668.48   9600       46
West        233  2855.56  10800       54


### Named aggregations

Named aggregations let you give each output column an explicit name, so the resulting DataFrame is self-explanatory without further renaming.

In [9]:
# Give each aggregation a clear name
summary = sales.groupby('region').agg(
    total_revenue=('revenue', 'sum'),
    avg_revenue=('revenue', 'mean'),
    total_units=('quantity', 'sum'),
    order_count=('order_id', 'count'),
).round(2)
print(summary)

        total_revenue  avg_revenue  total_units  order_count
region                                                      
East           155550      2880.56          275           54
North          110850      2409.78          229           46
South          122750      2668.48          226           46
West           154200      2855.56          233           54


The named-agg syntax is usually easier to read later than the dict-based version, especially when several columns are being summarized.

### Custom aggregation functions

If none of the built-in aggregations fit, any function that takes a Series and returns a single value will work. You can pass a regular function or a lambda.

In [10]:
# Apply a custom function
def revenue_range(s):
    return s.max() - s.min()

print(sales.groupby('region')['revenue'].agg(['mean', revenue_range]).round(2))

# Or with a lambda
print("\nUsing lambda:")
print(sales.groupby('region')['revenue'].agg(lambda s: s.max() - s.min()).round(2))

           mean  revenue_range
region                        
East    2880.56          10650
North   2409.78           9150
South   2668.48           9450
West    2855.56          10650

Using lambda:
region
East     10650
North     9150
South     9450
West     10650
Name: revenue, dtype: int64


## `transform()` and `filter()`

Aggregation reduces each group to one value. Sometimes that is not what is needed; the goal can be to keep every row but attach a group-level value to it, or to keep or drop entire groups based on some condition.

### definition

> "transform() returns an object that is indexed the same (same size) as the one being grouped." - pandas docs

The key difference between `agg` and `transform`:
- `agg()` returns one row per group.
- `transform()` returns one row per original row, with the group-level value broadcast back.

### `transform()` - put group-level values back onto each row

A common use is comparing each row to its group average (for example, how each order compares to its region's average).

In [11]:
# Add the region's average revenue as a new column on every row
sales['region_avg_revenue'] = sales.groupby('region')['revenue'].transform('mean')

print(sales[['region', 'revenue', 'region_avg_revenue']].head(10).round(2))

  region  revenue  region_avg_revenue
0   East     4000             2880.56
1   West      600             2855.56
2  North     2500             2409.78
3   East     1050             2880.56
4   East     4500             2880.56
5   West     5600             2855.56
6  North     4000             2409.78
7  North     1050             2409.78
8   East      750             2880.56
9  South     1500             2668.48


In [12]:
# Useful: how does each order compare to its region's average?
sales['vs_region_avg'] = sales['revenue'] - sales['region_avg_revenue']
print("Orders that beat their region's average:")
print(sales[sales['vs_region_avg'] > 0][['region', 'revenue', 'vs_region_avg']].head().round(2))

Orders that beat their region's average:
  region  revenue  vs_region_avg
0   East     4000        1119.44
2  North     2500          90.22
4   East     4500        1619.44
5   West     5600        2744.44
6  North     4000        1590.22


### `filter()` - keep or drop entire groups based on a condition

Where `transform` puts group info back onto rows, `filter` decides whether each entire group survives. The function passed to filter is called once per group and should return True (keep the group) or False (drop it).

In [13]:
# Only keep regions with > 50 orders
big_regions = sales.groupby('region').filter(lambda g: len(g) > 50)
print(f"Original orders: {len(sales)}")
print(f"After filter:    {len(big_regions)}")
print(f"Regions kept:    {big_regions['region'].unique()}")

Original orders: 200
After filter:    108
Regions kept:    ['East' 'West']


## Pivot Tables

### definition

> "Create a spreadsheet-style pivot table as a DataFrame. The levels in the pivot table will be stored in MultiIndex objects (hierarchical indexes) on the index and columns of the result DataFrame." - pandas docs

A **pivot table** is groupby followed by reshape in one call. If you have used pivot tables in Excel, this is the same operation in pandas: pick a column for rows, a column for columns, a column for values, and an aggregation to apply to those values.

In [14]:
# Total revenue by region (rows) and product (columns)
pt = sales.pivot_table(
    index='region',
    columns='product',
    values='revenue',
    aggfunc='sum'
).round(0)

print(pt)

product  Headphones  Laptop  Monitor  Phone  Tablet
region                                             
East          31150   35050    37750  27150   24450
North         33100   16600    16400   4550   40200
South          5700   32850    36000  21850   26350
West          23900   38100    33550  22900   35750


In [15]:
# Add totals (margins=True)
pt_totals = sales.pivot_table(
    index='region',
    columns='product',
    values='revenue',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
).round(0)

print(pt_totals)

product  Headphones  Laptop  Monitor  Phone  Tablet   Total
region                                                     
East          31150   35050    37750  27150   24450  155550
North         33100   16600    16400   4550   40200  110850
South          5700   32850    36000  21850   26350  122750
West          23900   38100    33550  22900   35750  154200
Total         93850  122600   123700  76450  126750  543350


In [16]:
# Multiple values + multiple aggregations
pt2 = sales.pivot_table(
    index='region',
    values=['revenue', 'quantity'],
    aggfunc=['sum', 'mean']
).round(2)

print(pt2)

            sum             mean         
       quantity revenue quantity  revenue
region                                   
East        275  155550     5.09  2880.56
North       229  110850     4.98  2409.78
South       226  122750     4.91  2668.48
West        233  154200     4.31  2855.56


### `crosstab` - a pivot table specialized for counting

> "Compute a simple cross tabulation of two (or more) factors. By default computes a frequency table of the factors unless an array of values and an aggregation function are passed." - pandas docs

`pd.crosstab` is shorthand for a pivot table whose aggregation is a count. Useful when the question is "how many rows match each combination of these two columns?"

In [17]:
# How many orders for each (region, product) combination?
print(pd.crosstab(sales['region'], sales['product']))

product  Headphones  Laptop  Monitor  Phone  Tablet
region                                             
East              8      13       13     11       9
North            14       9        6      4      13
South             3      13       14      8       8
West              5      16       13      6      14


In [18]:
# As percentages
print("As row percentages:")
print((pd.crosstab(sales['region'], sales['product'], normalize='index') * 100).round(1))

As row percentages:
product  Headphones  Laptop  Monitor  Phone  Tablet
region                                             
East           14.8    24.1     24.1   20.4    16.7
North          30.4    19.6     13.0    8.7    28.3
South           6.5    28.3     30.4   17.4    17.4
West            9.3    29.6     24.1   11.1    25.9


## Reshaping - `melt` and `pivot`

### definition

> "Unpivots a DataFrame from wide to long format. This function is useful to massage a DataFrame into a format where one or more columns are identifier variables, while all other columns are measured variables." - pandas docs

A few terms before the examples:
- **Wide format**: one column per category. For example, a `math`, `physics`, and `cs` column, each holding a score, with one row per student.
- **Long format**: one row per category. For example, one row per student-subject pair, with `subject` in one column and `score` in another.
- **Reshaping**: converting between wide and long.

### `melt` - wide to long

In [19]:
# Wide format example
wide = pd.DataFrame({
    'student': ['Ali', 'Sara', 'Ahmed'],
    'math':    [85, 92, 78],
    'physics': [88, 85, 82],
    'cs':      [95, 90, 88]
})
print("Wide format:")
print(wide)

Wide format:
  student  math  physics  cs
0     Ali    85       88  95
1    Sara    92       85  90
2   Ahmed    78       82  88


In [20]:
# Melt to long format
long = wide.melt(
    id_vars='student',
    var_name='subject',
    value_name='score'
)
print("Long format:")
print(long)

Long format:
  student  subject  score
0     Ali     math     85
1    Sara     math     92
2   Ahmed     math     78
3     Ali  physics     88
4    Sara  physics     85
5   Ahmed  physics     82
6     Ali       cs     95
7    Sara       cs     90
8   Ahmed       cs     88


Most plotting libraries and statistical methods expect long format, so going from wide to long is a routine step in real data work.

### `pivot` - long to wide (the reverse of melt)

> "Return reshaped DataFrame organized by given index / column values." - pandas docs

In [21]:
# Back to wide
back_to_wide = long.pivot(
    index='student',
    columns='subject',
    values='score'
)
print(back_to_wide)

subject  cs  math  physics
student                   
Ahmed    88    78       82
Ali      95    85       88
Sara     90    92       85


**Quick reference:**
- `melt()`: wide to long (unpivot)
- `pivot()`: long to wide, no aggregation (raises an error if there are duplicate index/column pairs)
- `pivot_table()`: long to wide, with aggregation (handles duplicates by reducing them with the chosen function)

## Merging and Joining DataFrames

### definition

> "Merge DataFrame or named Series objects with a database-style join. A named Series object is treated as a DataFrame with a single named column." - pandas docs

A few terms before the examples:
- **Merge / Join**: combining two DataFrames into one by matching rows on a shared column.
- **Key**: the column whose values are compared between the two tables to decide which rows match.
- **Inner / left / right / outer**: the four ways to decide what to do with rows whose keys do not have a match on the other side.

Joining DataFrames in pandas behaves the same way as JOIN in SQL.

### Setting up two related tables

In [22]:
# Table 1: orders
orders = pd.DataFrame({
    'order_id':    [101, 102, 103, 104, 105],
    'customer_id': [1, 2, 1, 3, 4],
    'amount':      [250, 175, 420, 60, 800]
})

# Table 2: customers
customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 5],
    'name':        ['Alice', 'Bob', 'Charlie', 'Diana'],
    'city':        ['NYC', 'LA', 'Chicago', 'NYC']
})

print("ORDERS:")
print(orders)
print("\nCUSTOMERS:")
print(customers)

ORDERS:
   order_id  customer_id  amount
0       101            1     250
1       102            2     175
2       103            1     420
3       104            3      60
4       105            4     800

CUSTOMERS:
   customer_id     name     city
0            1    Alice      NYC
1            2      Bob       LA
2            3  Charlie  Chicago
3            5    Diana      NYC


### Inner join (the default) - keep only matching rows

An **inner join** returns only the rows whose key value appears in both tables. Rows with no match on either side are dropped.

In [23]:
# inner join: only customer IDs that exist in BOTH tables
inner = orders.merge(customers, on='customer_id', how='inner')
print(inner)

# Notice: customer 4 from orders is dropped (not in customers)
# Notice: customer 5 from customers is dropped (no orders)

   order_id  customer_id  amount     name     city
0       101            1     250    Alice      NYC
1       102            2     175      Bob       LA
2       103            1     420    Alice      NYC
3       104            3      60  Charlie  Chicago


### Left join - keep all rows from the left DataFrame

A **left join** keeps every row from the left DataFrame and fills NaN into the right-side columns for any row whose key has no match on the right.

In [24]:
# left join: keep all orders, fill missing customer info with NaN
left = orders.merge(customers, on='customer_id', how='left')
print(left)

# Customer 4 is kept; name and city are NaN

   order_id  customer_id  amount     name     city
0       101            1     250    Alice      NYC
1       102            2     175      Bob       LA
2       103            1     420    Alice      NYC
3       104            3      60  Charlie  Chicago
4       105            4     800      NaN      NaN


### Right and outer joins

**Right join** is the mirror image of left join: it keeps every row from the right DataFrame and fills NaN where the left side has no match.

**Outer join** keeps every row from both tables, putting NaN in the columns of whichever side has no match.

In [25]:
# right join: keep all customers
right = orders.merge(customers, on='customer_id', how='right')
print("Right join:")
print(right)

print("\nOuter join (keep everything):")
outer = orders.merge(customers, on='customer_id', how='outer')
print(outer)

Right join:
   order_id  customer_id  amount     name     city
0     101.0            1   250.0    Alice      NYC
1     103.0            1   420.0    Alice      NYC
2     102.0            2   175.0      Bob       LA
3     104.0            3    60.0  Charlie  Chicago
4       NaN            5     NaN    Diana      NYC

Outer join (keep everything):
   order_id  customer_id  amount     name     city
0     101.0            1   250.0    Alice      NYC
1     103.0            1   420.0    Alice      NYC
2     102.0            2   175.0      Bob       LA
3     104.0            3    60.0  Charlie  Chicago
4     105.0            4   800.0      NaN      NaN
5       NaN            5     NaN    Diana      NYC


### Quick reference for joins

| Join type | Result |
|---|---|
| `inner` | Only rows with matching keys in both DataFrames |
| `left`  | All rows from the LEFT DataFrame, plus matching from right (NaN where no match) |
| `right` | All rows from the RIGHT DataFrame, plus matching from left |
| `outer` | All rows from both DataFrames |

In practice, `left` is the most common when adding information to a main table. `inner` is common when only clean intersections are wanted.

### Joining on different column names

If the key column has a different name in each DataFrame, use `left_on` and `right_on` to point to each side's column.

In [26]:
# Suppose customers had a column 'id' instead of 'customer_id'
customers_alt = customers.rename(columns={'customer_id': 'id'})

merged = orders.merge(
    customers_alt,
    left_on='customer_id',
    right_on='id',
    how='left'
)
print(merged)

   order_id  customer_id  amount   id     name     city
0       101            1     250  1.0    Alice      NYC
1       102            2     175  2.0      Bob       LA
2       103            1     420  1.0    Alice      NYC
3       104            3      60  3.0  Charlie  Chicago
4       105            4     800  NaN      NaN      NaN


## Concatenating - Stacking DataFrames

### definition

> "Concatenate pandas objects along a particular axis. Allows optional set logic along the other axes." - pandas docs

`concat` is for stacking DataFrames that already have the same structure (for example, monthly sales data spread across separate files). Unlike `merge`, no key column is used: rows or columns are simply stacked.

In [27]:
# Two DataFrames with the same columns
jan = pd.DataFrame({
    'month':  ['Jan', 'Jan', 'Jan'],
    'product':['A', 'B', 'C'],
    'sales':  [100, 200, 150]
})

feb = pd.DataFrame({
    'month':  ['Feb', 'Feb', 'Feb'],
    'product':['A', 'B', 'C'],
    'sales':  [120, 180, 175]
})

# Stack them vertically
combined = pd.concat([jan, feb], ignore_index=True)
print(combined)

  month product  sales
0   Jan       A    100
1   Jan       B    200
2   Jan       C    150
3   Feb       A    120
4   Feb       B    180
5   Feb       C    175


In [28]:
# Without ignore_index, the original indexes are kept (which is sometimes useful)
print(pd.concat([jan, feb]))

  month product  sales
0   Jan       A    100
1   Jan       B    200
2   Jan       C    150
0   Feb       A    120
1   Feb       B    180
2   Feb       C    175


### `concat` with axis=1 - stacking horizontally

With `axis=1`, `concat` lines DataFrames up side by side. Rows are matched on the index.

In [29]:
# Two DataFrames with the SAME index but different columns
left_df = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})
right_df = pd.DataFrame({'C': [7, 8, 9], 'D': [10, 11, 12]})

combined = pd.concat([left_df, right_df], axis=1)
print(combined)

   A  B  C   D
0  1  4  7  10
1  2  5  8  11
2  3  6  9  12


##  A Worked Analysis

This section uses the sales data from the start of the notebook to answer a few realistic questions, applying the tools from the earlier sections.

In [30]:
# 1. Which salesperson generates the most revenue?
top_sales = sales.groupby('salesperson').agg(
    total_revenue=('revenue', 'sum'),
    avg_order=('revenue', 'mean'),
    orders=('order_id', 'count'),
).round(2).sort_values('total_revenue', ascending=False)

print("Salesperson performance:")
print(top_sales)

Salesperson performance:
             total_revenue  avg_order  orders
salesperson                                  
Fatima              149050    3387.50      44
Hassan              103250    2346.59      44
Sara                 99000    2357.14      42
Ali                  97350    2433.75      40
Ahmed                94700    3156.67      30


In [31]:
# 2. Monthly trend per region
sales['month'] = sales['date'].dt.to_period('M')

monthly = sales.pivot_table(
    index='month',
    columns='region',
    values='revenue',
    aggfunc='sum'
).round(0)
print("Monthly revenue by region:")
print(monthly)

Monthly revenue by region:
region    East  North  South   West
month                              
2024-01  22200  13800   7400  28850
2024-02  27200  17400  12150  32350
2024-03  23000  19050  19500  23800
2024-04  25200   5400  43450  20150
2024-05  13150  22250   3550  28000
2024-06  22100  18600  18100  10250
2024-07  22700  14350  18600  10800


In [32]:
# 3. Each salesperson's best product
best_product = sales.groupby(['salesperson', 'product'])['revenue'].sum()
print(best_product.head(15))

# Get the top product per salesperson
top_per_person = best_product.groupby('salesperson').idxmax()
print("\nBest product per salesperson:")
print(top_per_person)

salesperson  product   
Ahmed        Headphones    20650
             Laptop        23750
             Monitor       17100
             Phone         11700
             Tablet        21500
Ali          Headphones    20200
             Laptop        18900
             Monitor       17950
             Phone         32800
             Tablet         7500
Fatima       Headphones    32950
             Laptop        38300
             Monitor       32500
             Phone         16550
             Tablet        28750
Name: revenue, dtype: int32

Best product per salesperson:
salesperson
Ahmed      (Ahmed, Laptop)
Ali           (Ali, Phone)
Fatima    (Fatima, Laptop)
Hassan    (Hassan, Tablet)
Sara        (Sara, Tablet)
Name: revenue, dtype: object


## Practice Exercises

In [33]:
# Exercise setup — two tables
np.random.seed(0)
students = pd.DataFrame({
    'student_id': range(1, 11),
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve',
             'Frank', 'Grace', 'Henry', 'Ivy', 'Jack'],
    'major': ['CS', 'EE', 'CS', 'ME', 'EE', 'CS', 'ME', 'CS', 'EE', 'ME'],
    'year': [2, 3, 4, 1, 2, 3, 4, 2, 1, 3]
})

grades = pd.DataFrame({
    'student_id': [1, 1, 1, 2, 2, 3, 3, 3, 4, 5, 5, 6, 7, 8, 9, 10],
    'subject':    ['Math', 'CS101', 'Physics', 'Math', 'EE201',
                   'Math', 'CS101', 'CS301', 'ME101', 'Math',
                   'EE201', 'CS101', 'ME101', 'CS301', 'EE201', 'ME101'],
    'grade':      [85, 92, 78, 70, 88, 95, 90, 87, 72, 80,
                   85, 91, 76, 89, 82, 74]
})

print("Students:")
print(students)
print("\nGrades:")
print(grades.head(10))

Students:
   student_id     name major  year
0           1    Alice    CS     2
1           2      Bob    EE     3
2           3  Charlie    CS     4
3           4    Diana    ME     1
4           5      Eve    EE     2
5           6    Frank    CS     3
6           7    Grace    ME     4
7           8    Henry    CS     2
8           9      Ivy    EE     1
9          10     Jack    ME     3

Grades:
   student_id  subject  grade
0           1     Math     85
1           1    CS101     92
2           1  Physics     78
3           2     Math     70
4           2    EE201     88
5           3     Math     95
6           3    CS101     90
7           3    CS301     87
8           4    ME101     72
9           5     Math     80


### Exercise 1
Find the average grade per subject.

### Exercise 2
For each student, find their average grade and number of subjects taken.

### Exercise 3
Merge the two tables to create a single DataFrame containing each grade with the student's name and major. Then find the average grade per major.

### Exercise 4
Create a pivot table with majors as rows and subjects as columns, showing average grades.

### Exercise 5 (challenge)
Find the top student (highest average grade) in each major. Hint: use groupby + sort + head.

In [34]:
# Exercise 1
print(grades.groupby('subject')['grade'].mean().round(2).sort_values(ascending=False))

subject
CS101      91.0
CS301      88.0
EE201      85.0
Math       82.5
Physics    78.0
ME101      74.0
Name: grade, dtype: float64


In [35]:
# Exercise 2
student_stats = grades.groupby('student_id').agg(
    avg_grade=('grade', 'mean'),
    subjects=('subject', 'count'),
).round(2)
print(student_stats)

            avg_grade  subjects
student_id                     
1               85.00         3
2               79.00         2
3               90.67         3
4               72.00         1
5               82.50         2
6               91.00         1
7               76.00         1
8               89.00         1
9               82.00         1
10              74.00         1


In [36]:
# Exercise 3
# Merge
combined = grades.merge(students, on='student_id')
print("Combined head:")
print(combined.head(8))

# Average grade per major
print("\nAverage grade per major:")
print(combined.groupby('major')['grade'].mean().round(2))

Combined head:
   student_id  subject  grade     name major  year
0           1     Math     85    Alice    CS     2
1           1    CS101     92    Alice    CS     2
2           1  Physics     78    Alice    CS     2
3           2     Math     70      Bob    EE     3
4           2    EE201     88      Bob    EE     3
5           3     Math     95  Charlie    CS     4
6           3    CS101     90  Charlie    CS     4
7           3    CS301     87  Charlie    CS     4

Average grade per major:
major
CS    88.38
EE    81.00
ME    74.00
Name: grade, dtype: float64


In [37]:
# Exercise 4
pt = combined.pivot_table(
    index='major',
    columns='subject',
    values='grade',
    aggfunc='mean'
).round(1)
print(pt)

subject  CS101  CS301  EE201  ME101  Math  Physics
major                                             
CS        91.0   88.0    NaN    NaN  90.0     78.0
EE         NaN    NaN   85.0    NaN  75.0      NaN
ME         NaN    NaN    NaN   74.0   NaN      NaN


In [38]:
# Exercise 5
# Average grade per student
avg = combined.groupby(['major', 'name'])['grade'].mean().round(2)

# Sort within each major and pick the top
top_per_major = avg.sort_values(ascending=False).groupby('major').head(1)
print("Top student per major:")
print(top_per_major)

Top student per major:
major  name 
CS     Frank    91.0
EE     Eve      82.5
ME     Grace    76.0
Name: grade, dtype: float64
